In [1]:
import sys
print(sys.executable)


/Users/ankitaupadhyaya/anaconda3/bin/python


In [3]:
!pip install tensorflow

  Obtaining dependency information for numpy<2.2.0,>=1.26.0 from https://files.pythonhosted.org/packages/47/7c/864cb966b96fce5e63fcf25e1e4d957fe5725a635e5f11fe03f39dd9d6b5/numpy-2.1.3-cp311-cp311-macosx_14_0_arm64.whl.metadata
  Using cached numpy-2.1.3-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached numpy-2.1.3-cp311-cp311-macosx_14_0_arm64.whl (5.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 2.1.3 which is incompatible.
scipy 1.11.1 requires nump

In [4]:
pip install numpy==1.24.4


  Obtaining dependency information for numpy==1.24.4 from https://files.pythonhosted.org/packages/c0/bc/77635c657a3668cf652806210b8662e1aff84b818a55ba88257abf6637a8/numpy-1.24.4-cp311-cp311-macosx_11_0_arm64.whl.metadata
  Using cached numpy-1.24.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp311-cp311-macosx_11_0_arm64.whl (13.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 1.24.4 which is incompatible.
Note: you may need to 

In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)


NumPy: 1.24.4
TensorFlow: 2.19.0


In [1]:
import os
import numpy as np
import librosa
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report 
import matplotlib.pyplot as plt


/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def extract_mfcc_sequences(folder_path, n_mfcc=13, duration=1.0):
    data = []
    labels = []
    
    for label, digit_folder in enumerate(sorted(os.listdir(folder_path))):
        digit_path = os.path.join(folder_path, digit_folder)
        
        if not os.path.isdir(digit_path):
            continue
        
        for file in os.listdir(digit_path):
            if not file.lower().endswith(".wav"):  # Skip non-wav files like .DS_Store
                continue

            file_path = os.path.join(digit_path, file)
            try:
                y, sr = librosa.load(file_path, duration=duration)
                mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T
                data.append(mfcc)
                labels.append(label)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    
    return data, labels


In [3]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# First, pad all MFCC sequences to the same shape
X_sequences_padded = pad_sequences(X_sequences, padding='post', dtype='float32')

# Now split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_sequences_padded, y, test_size=0.2, random_state=42, stratify=y)


NameError: name 'X_sequences' is not defined

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_sequences_padded = pad_sequences(X_sequences, padding='post', dtype='float32')
X_train, X_test, y_train, y_test = train_test_split(X_sequences_padded, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
X_train = np.array(X_train)
X_test = np.array(X_test)

# LSTM expects data of shape (samples, time_steps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical

# Automatically find the correct number of classes
num_classes = np.max(y_train) + 1  # this will be 101 if the max label is 100

# One-hot encode
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)


In [ ]:
print("Unique classes in y_train:", np.unique(y_train))
print("Max label:", np.max(y_train))


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Define the model
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))
model.add(LSTM(64))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))  # final layer for classification

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_data=(X_test, y_test))



In [10]:
from tensorflow.keras.models import load_model

# Save the trained model
model.save("NEPALI_LSTM_model.h5")


NameError: name 'model' is not defined

In [63]:
import os
import numpy as np
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 1. Extract MFCC features
def extract_mfcc_sequences(folder_path, n_mfcc=40, duration=1.0):
    data = []
    labels = []
    
    for digit_folder in sorted(os.listdir(folder_path)):
        digit_path = os.path.join(folder_path, digit_folder)
        
        if not os.path.isdir(digit_path):
            continue
        
        label = int(digit_folder) - 1  # Convert folder name (e.g., '99') to int and subtract 1 for 0-based labels
        
        for file in os.listdir(digit_path):
            if not file.lower().endswith(".wav"):
                continue

            file_path = os.path.join(digit_path, file)
            try:
                y, sr = librosa.load(file_path, duration=duration)
                mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T
                data.append(mfcc)
                labels.append(label)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    
    return data, labels


# Replace with your actual dataset path
folder_path = "/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files"
X_sequences, y = extract_mfcc_sequences(folder_path)

# 2. Pad sequences
X_sequences_padded = pad_sequences(X_sequences, padding='post', dtype='float32')

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences_padded, y, test_size=0.2, random_state=42, stratify=y
)


In [64]:
import numpy as np

# Convert to numpy arrays (if not already)
X_train = np.array(X_train)
X_test = np.array(X_test)

# Reshaping is not needed here because pad_sequences already gives you the right shape
print("X_train shape:", X_train.shape)  # Should be (num_samples, time_steps, n_mfcc)
print("X_test shape:", X_test.shape)


X_train shape: (1437, 44, 40)
X_test shape: (360, 44, 40)


In [65]:
from tensorflow.keras.utils import to_categorical

num_classes = np.max(y_train) + 1  # Automatically detect number of classes

y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


y_train shape: (1437, 100)
y_test shape: (360, 100)


In [66]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()
model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(32))
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/200


/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.0066 - loss: 4.6284 - val_accuracy: 0.0167 - val_loss: 4.5829
Epoch 2/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0249 - loss: 4.5626 - val_accuracy: 0.0333 - val_loss: 4.4983
Epoch 3/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0308 - loss: 4.4289 - val_accuracy: 0.0333 - val_loss: 4.3309
Epoch 4/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0366 - loss: 4.2467 - val_accuracy: 0.0444 - val_loss: 4.1276
Epoch 5/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0604 - loss: 4.0693 - val_accuracy: 0.0611 - val_loss: 3.9605
Epoch 6/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.0661 - loss: 3.9020 - val_accuracy: 0.0806 - val_loss: 3.7791
Epoch 7/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.0804 - loss: 3.7237 - val_accuracy: 0.1083 - val_loss: 3.6433
Epoch 8/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.0977 - loss: 3.5950 - val_accuracy: 0.1139 - val_l

In [67]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Predict on test set
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Report
print("Accuracy: {:.2f}%".format(accuracy_score(y_true, y_pred) * 100))
print("Classification Report:\n", classification_report(y_true, y_pred))


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Accuracy: 68.61%
Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.67      0.67         3
           1       1.00      0.25      0.40         4
           2       0.67      1.00      0.80         4
           3       0.50      0.50      0.50         4
           4       1.00      1.00      1.00         3
           5       0.50      0.33      0.40         3
           6       0.50      0.50      0.50         4
           7       1.00      0.75      0.86         4
           8       1.00      0.75      0.86         4
           9       1.00      1.00      1.00         3
          10       0.50      0.67      0.57         3
          11       1.00      0.33      0.50         3
          12       1.00      0.33      0.50         3
          13       0.60      0.75      0.67         4
          14       0.67      1.00      0.80         4
          15       1.00      0.33      0.50         3


/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

In [83]:
import joblib

# Save the trained model
model.save("NEPALI_LSTM_model.keras")


# No scaler used here, but if you had one (e.g., StandardScaler), you could save it like:
# joblib.dump(scaler, "NEPALI_LSTM_scaler.joblib")


In [85]:
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def extract_mfcc_from_file(file_path, n_mfcc=40, duration=1.0):
    y, sr = librosa.load(file_path, duration=duration)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T
    return mfcc


In [87]:
label_names = [str(i) for i in range(1, 101)]  # 1 to 100

In [93]:
import numpy as np
from tensorflow.keras.models import load_model
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences

def extract_mfcc_from_file(file_path, n_mfcc=40, duration=1.0):
    y, sr = librosa.load(file_path, duration=duration)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T
    return mfcc

def predict_digit(file_path, model, label_names):
    mfcc_seq = extract_mfcc_from_file(file_path, n_mfcc=40)
    padded_seq = pad_sequences([mfcc_seq], padding='post', dtype='float32')
    
    prediction = model.predict(padded_seq)
    predicted_index = np.argmax(prediction)
    predicted_label = label_names[predicted_index]  # Maps back to '1' to '100'

    print(f"Predicted class index: {predicted_index}")
    print(f"Predicted label (digit): {predicted_label}")
    return predicted_index, predicted_label


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [95]:
model = load_model("NEPALI_LSTM_model.keras")
file_path = "/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files/25/NEP_S2_D(25)_r1.wav"

_, label = predict_digit(file_path, model, label_names)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
Predicted class index: 37
Predicted label (digit): 38
